[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/01_collection/A7_comprehensive_integration.ipynb)

# A7: Comprehensive Data Integration

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Perform fuzzy string matching** to link records across datasets
2. **Merge datasets** with different schemas and coverage
3. **Handle match quality tiers** (confirmed, possible, no match)
4. **Generate integration reports** summarizing data quality

## Why This Matters

Real-world data integration is messy:
- **Official data** (permits): Authoritative but limited scope
- **Community data** (Gellerman): Broader coverage but unofficial
- **Challenge**: Same project, different names ("2700 Shattuck" vs "2700 Shattuck Avenue")

Fuzzy matching allows us to connect these datasets despite imperfect alignment.

---

## Overview

Integrate official permit data with community-sourced Gellerman data.

**Inputs:**
- `data/processed/housing_projects_FINAL.csv` (115 official projects)
- `outputs/gellerman_raw.csv` (~203 community projects)
- `outputs/gellerman_news_links.csv` (news URLs)

**Outputs:**
- `outputs/housing_projects_enriched.csv` (115 official + news links)
- `outputs/housing_projects_comprehensive.csv` (~203 combined)
- `outputs/match_results.csv` (matching details)
- `outputs/gellerman_integration_report.html` (analysis)

---

## 1. Setup & Environment

In [1]:
# ============================================================================
# COLAB ENVIRONMENT SETUP
# ============================================================================

import os
import sys
from pathlib import Path

print('Setting up environment...')
print('='*70)

# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

if IN_COLAB:
    repo_path = Path('/content/berkeley-housing-analysis')
    
    if not repo_path.exists():
        print('\nCloning repository...')
        !git clone https://github.com/blockXblock/berkeley-housing-analysis.git
        print('Repository cloned')
    else:
        print('\nRepository already exists')
        !cd /content/berkeley-housing-analysis && git pull
    
    os.chdir(repo_path)
    
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))
    
    (repo_path / 'outputs').mkdir(exist_ok=True)
    
    ROOT = repo_path
    
    # Install fuzzy matching library
    print('\nInstalling fuzzy matching library...')
    !pip install -q fuzzywuzzy python-Levenshtein
else:
    def find_project_root():
        current = Path.cwd()
        for path in [current] + list(current.parents):
            if (path / '00_config').exists() or (path / 'data').exists():
                return path
        return current
    
    ROOT = find_project_root()
    os.chdir(ROOT)

(ROOT / 'outputs').mkdir(exist_ok=True)

print(f'\nWorking directory: {os.getcwd()}')
print('='*70)

Setting up environment...
Running locally

Working directory: /Users/johngage/berkeley-data


In [2]:
# ============================================================================
# INSTALL DEPENDENCIES
# ============================================================================

# Try to import fuzzywuzzy, install if not available
try:
    from fuzzywuzzy import fuzz, process
    print('fuzzywuzzy library loaded')
except ImportError:
    print('Installing fuzzywuzzy...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'fuzzywuzzy', 'python-Levenshtein'])
    from fuzzywuzzy import fuzz, process
    print('fuzzywuzzy installed and loaded')

Installing fuzzywuzzy...
fuzzywuzzy installed and loaded


In [3]:
# ============================================================================
# IMPORTS
# ============================================================================

import pandas as pd
import numpy as np
from datetime import datetime
import time
import json

print('All imports loaded successfully')

All imports loaded successfully


In [4]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

_cell_start_time = None

def timer_start(label=""):
    global _cell_start_time
    _cell_start_time = time.time()
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    if label:
        print(f'{label}')
    print(f'Started: {now}')
    print('='*70)

def timer_end():
    global _cell_start_time
    duration = time.time() - _cell_start_time
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print('='*70)
    print(f'Completed: {now}')
    print(f'Duration: {duration:.2f} seconds')

print('Timestamp utilities loaded')

Timestamp utilities loaded


## 2. Load Data Sources

In [5]:
# ============================================================================
# LOAD OFFICIAL PROJECTS
# ============================================================================

timer_start('LOADING OFFICIAL PROJECTS')

official_path = ROOT / 'data/processed/housing_projects_FINAL.csv'

if official_path.exists():
    df_official = pd.read_csv(official_path)
    print(f'Loaded: {official_path}')
    print(f'  Rows: {len(df_official)}')
    print(f'  Columns: {len(df_official.columns)}')
    
    # Key columns
    print(f'\nKey columns:')
    for col in ['address_display', 'address_norm', 'net_units', 'status', 'latitude', 'longitude']:
        if col in df_official.columns:
            non_null = df_official[col].notna().sum()
            print(f'  {col}: {non_null}/{len(df_official)} non-null')
    
    # Sample
    print('\nSample data:')
    display(df_official[['address_display', 'net_units', 'status']].head(5))
else:
    print(f'ERROR: File not found: {official_path}')
    df_official = pd.DataFrame()

timer_end()

LOADING OFFICIAL PROJECTS
Started: 2026-02-26 19:57:43
Loaded: /Users/johngage/berkeley-data/data/processed/housing_projects_FINAL.csv
  Rows: 115
  Columns: 25

Key columns:
  address_display: 115/115 non-null
  address_norm: 115/115 non-null
  net_units: 115/115 non-null
  status: 115/115 non-null
  latitude: 115/115 non-null
  longitude: 115/115 non-null

Sample data:


,address_display,net_units,status
0,1750 SACRAMENTO St,739.0,Under Review
1,2276 SHATTUCK Ave,336.0,In Review
2,2700 SHATTUCK Ave,276.0,In Review
3,1914 FIFTH St,257.0,Under Review
4,2425 DURANT Ave,250.0,Pending Final Action


Completed: 2026-02-26 19:57:43
Duration: 0.02 seconds


In [6]:
# ============================================================================
# LOAD GELLERMAN DATA (from A6)
# ============================================================================

timer_start('LOADING GELLERMAN DATA')

gellerman_path = ROOT / 'outputs/gellerman_raw.csv'
news_path = ROOT / 'outputs/gellerman_news_links.csv'

# Load raw projects
if gellerman_path.exists():
    df_gellerman = pd.read_csv(gellerman_path)
    print(f'Loaded: {gellerman_path}')
    print(f'  Rows: {len(df_gellerman)}')
else:
    print(f'WARNING: Gellerman data not found at {gellerman_path}')
    print('Run A6_community_map_import.ipynb first')
    
    # Create empty DataFrame with expected columns
    df_gellerman = pd.DataFrame(columns=['name', 'address_normalized', 'latitude', 'longitude', 'coords_valid', 'url_count'])

# Load news links
if news_path.exists():
    df_news = pd.read_csv(news_path)
    print(f'\nLoaded: {news_path}')
    print(f'  URLs: {len(df_news)}')
    
    # Source breakdown
    if 'source_category' in df_news.columns:
        print('\nURLs by source:')
        for source, count in df_news['source_category'].value_counts().head(5).items():
            print(f'  {source}: {count}')
else:
    print(f'\nWARNING: News links not found at {news_path}')
    df_news = pd.DataFrame(columns=['project_name', 'url', 'source_category', 'address_normalized'])

timer_end()

LOADING GELLERMAN DATA
Started: 2026-02-26 19:57:43
Loaded: /Users/johngage/berkeley-data/outputs/gellerman_raw.csv
  Rows: 207

Loaded: /Users/johngage/berkeley-data/outputs/gellerman_news_links.csv
  URLs: 2024

URLs by source:
  Other: 1583
  SFYimby: 267
  Berkeleyside: 96
  City of Berkeley: 57
  Daily Cal: 11
Completed: 2026-02-26 19:57:43
Duration: 0.02 seconds


## 3. Fuzzy Matching

### Key Concept: String Similarity

**Fuzzy matching** finds similar strings even when they're not identical:
- "2700 SHATTUCK AVE" vs "2700 Shattuck Avenue" → 95% similar
- Uses **Levenshtein distance** (edit distance) under the hood

**Matching thresholds:**
- **≥85**: Confirmed match (high confidence)
- **70-84**: Possible match (needs review)
- **<70**: No match (different projects)

In [7]:
# ============================================================================
# FUZZY MATCHING CONFIGURATION
# ============================================================================

# Matching thresholds
THRESHOLD_CONFIRMED = 85  # High confidence match
THRESHOLD_POSSIBLE = 70   # Needs manual review

def categorize_match_score(score):
    """
    Categorize a match score.
    
    Returns: 'confirmed', 'possible', or 'no_match'
    """
    if score >= THRESHOLD_CONFIRMED:
        return 'confirmed'
    elif score >= THRESHOLD_POSSIBLE:
        return 'possible'
    else:
        return 'no_match'

print(f'Match thresholds:')
print(f'  Confirmed: >= {THRESHOLD_CONFIRMED}')
print(f'  Possible:  >= {THRESHOLD_POSSIBLE}')
print(f'  No match:  < {THRESHOLD_POSSIBLE}')

Match thresholds:
  Confirmed: >= 85
  Possible:  >= 70
  No match:  < 70


In [8]:
# ============================================================================
# FUZZY MATCHING FUNCTION
# ============================================================================

def fuzzy_match_addresses(source_addresses, target_addresses, threshold=70):
    """
    Fuzzy match source addresses to target addresses.
    
    Parameters:
    -----------
    source_addresses : list
        Addresses to match (e.g., Gellerman)
    target_addresses : list
        Addresses to match against (e.g., official)
    threshold : int
        Minimum score to consider a match
        
    Returns:
    --------
    list of dict
        Match results with scores
    """
    results = []
    
    # Filter out None values from targets
    valid_targets = [t for t in target_addresses if t and pd.notna(t)]
    
    for source in source_addresses:
        if not source or pd.isna(source):
            results.append({
                'source_address': source,
                'best_match': None,
                'score': 0,
                'category': 'no_match'
            })
            continue
        
        # Find best match
        if valid_targets:
            best_match, score = process.extractOne(
                source, 
                valid_targets,
                scorer=fuzz.token_sort_ratio  # Good for address matching
            )
        else:
            best_match, score = None, 0
        
        results.append({
            'source_address': source,
            'best_match': best_match if score >= threshold else None,
            'score': score,
            'category': categorize_match_score(score)
        })
    
    return results

print('Fuzzy matching function loaded')

Fuzzy matching function loaded


In [9]:
# ============================================================================
# PERFORM FUZZY MATCHING
# ============================================================================

timer_start('FUZZY MATCHING GELLERMAN TO OFFICIAL')

if len(df_gellerman) > 0 and len(df_official) > 0:
    # Get addresses for matching
    gellerman_addresses = df_gellerman['address_normalized'].tolist()
    
    # Use normalized address from official data
    if 'address_norm' in df_official.columns:
        official_addresses = df_official['address_norm'].tolist()
    else:
        official_addresses = df_official['address_display'].tolist()
    
    print(f'Matching {len(gellerman_addresses)} Gellerman addresses')
    print(f'Against {len(official_addresses)} official addresses')
    print(f'\nThis may take a moment...')
    
    # Perform matching
    match_results = fuzzy_match_addresses(
        gellerman_addresses, 
        official_addresses,
        threshold=THRESHOLD_POSSIBLE
    )
    
    # Create results DataFrame
    df_matches = pd.DataFrame(match_results)
    
    # Add Gellerman project names
    df_matches['gellerman_name'] = df_gellerman['name'].values
    df_matches['gellerman_lat'] = df_gellerman['latitude'].values
    df_matches['gellerman_lon'] = df_gellerman['longitude'].values
    
    # Match statistics
    print(f'\nMatch Results:')
    category_counts = df_matches['category'].value_counts()
    for cat, count in category_counts.items():
        pct = 100 * count / len(df_matches)
        print(f'  {cat}: {count} ({pct:.1f}%)')
    
    # Score distribution
    print(f'\nScore statistics:')
    print(f'  Mean: {df_matches["score"].mean():.1f}')
    print(f'  Median: {df_matches["score"].median():.1f}')
    print(f'  Max: {df_matches["score"].max()}')
    
    # Sample matches
    print('\nSample confirmed matches:')
    confirmed = df_matches[df_matches['category'] == 'confirmed'].head(5)
    display(confirmed[['gellerman_name', 'best_match', 'score']])
else:
    print('Insufficient data for matching')
    df_matches = pd.DataFrame()

timer_end()

FUZZY MATCHING GELLERMAN TO OFFICIAL
Started: 2026-02-26 19:57:43
Matching 207 Gellerman addresses
Against 115 official addresses

This may take a moment...

Match Results:
  confirmed: 117 (56.5%)
  possible: 49 (23.7%)
  no_match: 41 (19.8%)

Score statistics:
  Mean: 75.9
  Median: 86.0
  Max: 100

Sample confirmed matches:


,gellerman_name,best_match,score
1,1110 University Ave,1710 UNIVERSITY AVE,95
3,1367 University Ave,1790 UNIVERSITY AVE,89
4,1498 University Ave,1581 UNIVERSITY AVE,89
5,1581 University Ave,1581 UNIVERSITY AVE,100
6,1598 University Ave,1581 UNIVERSITY AVE,95


Completed: 2026-02-26 19:57:43
Duration: 0.09 seconds


## 4. Create Enriched Dataset

Add news coverage data to the official 115 projects.

In [10]:
# ============================================================================
# AGGREGATE NEWS LINKS BY PROJECT
# ============================================================================

timer_start('AGGREGATING NEWS LINKS')

if len(df_news) > 0:
    # News sources only (exclude city/permit links)
    news_sources = ['Berkeleyside', 'SFYimby', 'SF Chronicle', 'Daily Cal', 'SFGate', 'Mercury News', 'East Bay Times']
    df_news_only = df_news[df_news['source_category'].isin(news_sources)].copy()
    
    print(f'Total URLs: {len(df_news)}')
    print(f'News articles only: {len(df_news_only)}')
    
    # Aggregate by address
    if 'address_normalized' in df_news_only.columns:
        news_agg = df_news_only.groupby('address_normalized').agg({
            'url': lambda x: ';'.join(x.dropna().unique()),
            'source_category': lambda x: x.mode().iloc[0] if len(x) > 0 else None,
            'project_name': 'first'
        }).reset_index()
        
        news_agg.columns = ['address_normalized', 'news_urls', 'primary_news_source', 'gellerman_name']
        news_agg['num_news_articles'] = news_agg['news_urls'].apply(lambda x: len(x.split(';')) if x else 0)
        
        print(f'\nAggregated to {len(news_agg)} unique addresses with news coverage')
        
        # Sample
        print('\nSample aggregated news:')
        display(news_agg[['address_normalized', 'num_news_articles', 'primary_news_source']].head(5))
    else:
        news_agg = pd.DataFrame()
else:
    print('No news data available')
    news_agg = pd.DataFrame()

timer_end()

AGGREGATING NEWS LINKS
Started: 2026-02-26 19:57:43
Total URLs: 2024
News articles only: 383

Aggregated to 143 unique addresses with news coverage

Sample aggregated news:


,address_normalized,num_news_articles,primary_news_source
0,1050 MONROE ST,1,SFYimby
1,1099 ASHBY AVE,1,SFYimby
2,1130 OXFORD ST,2,Berkeleyside
3,1200 SAN PABLO AVE,1,Berkeleyside
4,1201 SAN PABLO AVE,1,SFYimby


Completed: 2026-02-26 19:57:43
Duration: 0.02 seconds


In [11]:
# ============================================================================
# CREATE ENRICHED OFFICIAL DATASET
# ============================================================================

timer_start('CREATING ENRICHED DATASET')

if len(df_official) > 0:
    # Start with official data
    df_enriched = df_official.copy()
    
    # Initialize new columns
    df_enriched['has_media_coverage'] = False
    df_enriched['num_news_articles'] = 0
    df_enriched['primary_news_source'] = None
    df_enriched['news_urls'] = None
    df_enriched['gellerman_match_score'] = None
    
    # Match column for joining
    match_col = 'address_norm' if 'address_norm' in df_enriched.columns else 'address_display'
    
    # Create lookup from matches (Gellerman address -> Official address -> Match info)
    if len(df_matches) > 0:
        confirmed_matches = df_matches[df_matches['category'] == 'confirmed'].copy()
        
        # For each confirmed match, link to news
        for _, match in confirmed_matches.iterrows():
            official_addr = match['best_match']
            gellerman_addr = match['source_address']
            score = match['score']
            
            # Find matching official project
            mask = df_enriched[match_col] == official_addr
            
            if mask.any():
                # Update match score
                df_enriched.loc[mask, 'gellerman_match_score'] = score
                
                # Find news for this Gellerman address
                if len(news_agg) > 0 and gellerman_addr in news_agg['address_normalized'].values:
                    news_row = news_agg[news_agg['address_normalized'] == gellerman_addr].iloc[0]
                    df_enriched.loc[mask, 'has_media_coverage'] = True
                    df_enriched.loc[mask, 'num_news_articles'] = news_row['num_news_articles']
                    df_enriched.loc[mask, 'primary_news_source'] = news_row['primary_news_source']
                    df_enriched.loc[mask, 'news_urls'] = news_row['news_urls']
    
    # Statistics
    print(f'Enriched dataset: {len(df_enriched)} rows')
    print(f'\nMedia coverage:')
    print(f'  With coverage: {df_enriched["has_media_coverage"].sum()}')
    print(f'  Without: {(~df_enriched["has_media_coverage"]).sum()}')
    print(f'  Total articles linked: {df_enriched["num_news_articles"].sum()}')
    
    print(f'\nGellerman match scores:')
    print(f'  Projects matched: {df_enriched["gellerman_match_score"].notna().sum()}')
    print(f'  Average score: {df_enriched["gellerman_match_score"].mean():.1f}')
    
    # Sample
    print('\nSample enriched projects with coverage:')
    with_coverage = df_enriched[df_enriched['has_media_coverage']]
    display(with_coverage[['address_display', 'num_news_articles', 'primary_news_source']].head(5))
else:
    print('No official data to enrich')
    df_enriched = pd.DataFrame()

timer_end()

CREATING ENRICHED DATASET
Started: 2026-02-26 19:57:43
Enriched dataset: 115 rows

Media coverage:
  With coverage: 47
  Without: 68
  Total articles linked: 130

Gellerman match scores:
  Projects matched: 50
  Average score: 92.8

Sample enriched projects with coverage:


,address_display,num_news_articles,primary_news_source
1,2276 SHATTUCK Ave,3,Berkeleyside
2,2700 SHATTUCK Ave,4,SFYimby
4,2425 DURANT Ave,1,Berkeleyside
5,2029 UNIVERSITY Ave,4,Berkeleyside
6,2601 SAN PABLO Ave,4,SFYimby


Completed: 2026-02-26 19:57:43
Duration: 0.07 seconds


## 5. Create Comprehensive Dataset

Combine official (115) + Gellerman-only (~88) projects.

In [12]:
# ============================================================================
# IDENTIFY GELLERMAN-ONLY PROJECTS
# ============================================================================

timer_start('IDENTIFYING NEW PROJECTS FROM GELLERMAN')

if len(df_matches) > 0 and len(df_gellerman) > 0:
    # Find projects that didn't match anything in official data
    no_match = df_matches[df_matches['category'] == 'no_match'].copy()
    
    # Get original Gellerman data for these
    gellerman_only_names = no_match['gellerman_name'].tolist()
    df_gellerman_only = df_gellerman[df_gellerman['name'].isin(gellerman_only_names)].copy()
    
    print(f'Gellerman-only projects (not in official data): {len(df_gellerman_only)}')
    
    if len(df_gellerman_only) > 0:
        # Show sample
        print('\nSample new discoveries:')
        display(df_gellerman_only[['name', 'address_normalized', 'url_count']].head(10))
else:
    print('No match data available')
    df_gellerman_only = pd.DataFrame()

timer_end()

IDENTIFYING NEW PROJECTS FROM GELLERMAN
Started: 2026-02-26 19:57:43
Gellerman-only projects (not in official data): 41

Sample new discoveries:


,name,address_normalized,url_count
0,North Berkeley BART station,NaN,35
19,2099 Martin Luther King,2099 MARTIN LUTHER KING,10
28,2037 Kala Bagai Wy,2037 KALA BAGAI WY,2
34,UC Berkeley Innovation Zone,NaN,34
36,130-134 Berkeley Square,NaN,9
39,The Gateway,NaN,19
42,Upper Hearst,NaN,9
43,Bechtel Engineering Center,NaN,15
44,Heathcock Hall,NaN,8
65,2450-2480 Shattuck Ave.,NaN,5


Completed: 2026-02-26 19:57:43
Duration: 0.01 seconds


In [13]:
# ============================================================================
# CREATE COMPREHENSIVE DATASET
# ============================================================================

timer_start('CREATING COMPREHENSIVE DATASET')

comprehensive_records = []
project_id = 1

# Add official projects
if len(df_enriched) > 0:
    for _, row in df_enriched.iterrows():
        record = {
            'project_id': project_id,
            'address_display': row.get('address_display'),
            'address_normalized': row.get('address_norm'),
            'latitude': row.get('latitude'),
            'longitude': row.get('longitude'),
            'data_source': 'both' if row.get('gellerman_match_score') else 'official_permit',
            'net_units': row.get('net_units'),
            'status': row.get('status'),
            'has_official_permit': True,
            'has_media_coverage': row.get('has_media_coverage', False),
            'news_urls': row.get('news_urls'),
            'primary_news_source': row.get('primary_news_source'),
            'gellerman_original_name': None,
            'gellerman_match_score': row.get('gellerman_match_score')
        }
        comprehensive_records.append(record)
        project_id += 1
    
    print(f'Added {len(df_enriched)} official projects')

# Add Gellerman-only projects
if len(df_gellerman_only) > 0:
    for _, row in df_gellerman_only.iterrows():
        # Get news for this project
        addr = row.get('address_normalized')
        news_data = {}
        if len(news_agg) > 0 and addr in news_agg['address_normalized'].values:
            news_row = news_agg[news_agg['address_normalized'] == addr].iloc[0]
            news_data = {
                'news_urls': news_row.get('news_urls'),
                'primary_news_source': news_row.get('primary_news_source')
            }
        
        record = {
            'project_id': project_id,
            'address_display': row.get('name'),
            'address_normalized': addr,
            'latitude': row.get('latitude'),
            'longitude': row.get('longitude'),
            'data_source': 'media_reported',
            'net_units': None,
            'status': 'Reported in media',
            'has_official_permit': False,
            'has_media_coverage': row.get('url_count', 0) > 0,
            'news_urls': news_data.get('news_urls'),
            'primary_news_source': news_data.get('primary_news_source'),
            'gellerman_original_name': row.get('name'),
            'gellerman_match_score': None
        }
        comprehensive_records.append(record)
        project_id += 1
    
    print(f'Added {len(df_gellerman_only)} Gellerman-only projects')

# Create DataFrame
df_comprehensive = pd.DataFrame(comprehensive_records)

print(f'\nComprehensive dataset: {len(df_comprehensive)} total projects')

# Breakdown
print(f'\nBy data source:')
for source, count in df_comprehensive['data_source'].value_counts().items():
    print(f'  {source}: {count}')

print(f'\nBy permit status:')
print(f'  Has official permit: {df_comprehensive["has_official_permit"].sum()}')
print(f'  Media-reported only: {(~df_comprehensive["has_official_permit"]).sum()}')

timer_end()

CREATING COMPREHENSIVE DATASET
Started: 2026-02-26 19:57:43
Added 115 official projects
Added 41 Gellerman-only projects

Comprehensive dataset: 156 total projects

By data source:
  official_permit: 65
  both: 50
  media_reported: 41

By permit status:
  Has official permit: 115
  Media-reported only: 41
Completed: 2026-02-26 19:57:43
Duration: 0.01 seconds


## 6. Generate Integration Report

In [14]:
# ============================================================================
# GENERATE HTML REPORT
# ============================================================================

timer_start('GENERATING INTEGRATION REPORT')

# Collect statistics
stats = {
    'generated_at': datetime.now().isoformat(),
    'official_count': len(df_enriched) if len(df_enriched) > 0 else 0,
    'gellerman_count': len(df_gellerman) if len(df_gellerman) > 0 else 0,
    'comprehensive_count': len(df_comprehensive) if len(df_comprehensive) > 0 else 0,
    'new_discoveries': len(df_gellerman_only) if len(df_gellerman_only) > 0 else 0,
}

if len(df_matches) > 0:
    stats['match_confirmed'] = (df_matches['category'] == 'confirmed').sum()
    stats['match_possible'] = (df_matches['category'] == 'possible').sum()
    stats['match_none'] = (df_matches['category'] == 'no_match').sum()

if len(df_news) > 0:
    stats['total_urls'] = len(df_news)
    stats['news_by_source'] = df_news['source_category'].value_counts().to_dict()

# Generate HTML
html_report = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Gellerman Integration Report</title>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 900px; margin: 0 auto; padding: 20px; }}
        h1 {{ color: #1a365d; }}
        h2 {{ color: #2c5282; border-bottom: 2px solid #e2e8f0; padding-bottom: 8px; }}
        .stat-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 15px; margin: 20px 0; }}
        .stat-card {{ background: #f7fafc; padding: 15px; border-radius: 8px; text-align: center; }}
        .stat-value {{ font-size: 2rem; font-weight: bold; color: #2c5282; }}
        .stat-label {{ color: #666; font-size: 0.9rem; }}
        table {{ width: 100%; border-collapse: collapse; margin: 15px 0; }}
        th, td {{ padding: 10px; text-align: left; border-bottom: 1px solid #e2e8f0; }}
        th {{ background: #f7fafc; color: #1a365d; }}
        .badge {{ display: inline-block; padding: 3px 8px; border-radius: 4px; font-size: 0.8rem; }}
        .badge-green {{ background: #c6f6d5; color: #22543d; }}
        .badge-yellow {{ background: #fefcbf; color: #744210; }}
        .badge-red {{ background: #fed7d7; color: #822727; }}
    </style>
</head>
<body>
    <h1>Gellerman Data Integration Report</h1>
    <p>Generated: {stats['generated_at']}</p>
    
    <h2>1. Executive Summary</h2>
    <div class="stat-grid">
        <div class="stat-card">
            <div class="stat-value">{stats['official_count']}</div>
            <div class="stat-label">Official Projects</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">{stats['gellerman_count']}</div>
            <div class="stat-label">Gellerman Projects</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">{stats['comprehensive_count']}</div>
            <div class="stat-label">Combined Total</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">{stats['new_discoveries']}</div>
            <div class="stat-label">New Discoveries</div>
        </div>
    </div>
    
    <h2>2. Match Quality Statistics</h2>
    <table>
        <tr><th>Category</th><th>Count</th><th>Description</th></tr>
        <tr><td><span class="badge badge-green">Confirmed</span></td><td>{stats.get('match_confirmed', 0)}</td><td>High confidence match (score ≥85)</td></tr>
        <tr><td><span class="badge badge-yellow">Possible</span></td><td>{stats.get('match_possible', 0)}</td><td>May need manual review (score 70-84)</td></tr>
        <tr><td><span class="badge badge-red">No Match</span></td><td>{stats.get('match_none', 0)}</td><td>New projects not in official data</td></tr>
    </table>
"""

# Add new discoveries table
if len(df_gellerman_only) > 0:
    html_report += """
    <h2>3. New Projects Discovered</h2>
    <p>These projects appear in Gellerman's map but not in official permit data:</p>
    <table>
        <tr><th>Project</th><th>Coordinates</th><th>News Links</th></tr>
"""
    for _, row in df_gellerman_only.head(20).iterrows():
        coords = f"({row['latitude']:.4f}, {row['longitude']:.4f})" if pd.notna(row.get('latitude')) else 'N/A'
        html_report += f"        <tr><td>{row['name']}</td><td>{coords}</td><td>{row.get('url_count', 0)}</td></tr>\n"
    
    if len(df_gellerman_only) > 20:
        html_report += f"        <tr><td colspan='3'><em>... and {len(df_gellerman_only) - 20} more</em></td></tr>\n"
    html_report += "    </table>\n"

# Add media source analysis
if 'news_by_source' in stats:
    html_report += """
    <h2>4. Media Source Analysis</h2>
    <table>
        <tr><th>Source</th><th>URLs</th></tr>
"""
    for source, count in sorted(stats['news_by_source'].items(), key=lambda x: -x[1]):
        html_report += f"        <tr><td>{source}</td><td>{count}</td></tr>\n"
    html_report += "    </table>\n"

# Close HTML
html_report += """
    <h2>5. Recommendations</h2>
    <ul>
        <li><strong>Verify possible matches:</strong> Review projects with match scores 70-84</li>
        <li><strong>Research new discoveries:</strong> Investigate Gellerman-only projects for permit status</li>
        <li><strong>Update coverage gaps:</strong> Official projects lacking news coverage may need outreach</li>
        <li><strong>Periodic refresh:</strong> Re-run integration monthly to capture new projects</li>
    </ul>
    
    <p style="margin-top: 40px; color: #666; font-size: 0.9rem;">
        Data sources: City of Berkeley Open Data Portal, Eric Gellerman's Berkeley Development Map
    </p>
</body>
</html>
"""

# Save report
report_path = ROOT / 'outputs/gellerman_integration_report.html'
with open(report_path, 'w') as f:
    f.write(html_report)

print(f'Saved: {report_path}')
print(f'Open in browser to view the report')

timer_end()

GENERATING INTEGRATION REPORT
Started: 2026-02-26 19:57:43
Saved: /Users/johngage/berkeley-data/outputs/gellerman_integration_report.html
Open in browser to view the report
Completed: 2026-02-26 19:57:43
Duration: 0.00 seconds


## 7. Export All Data

In [15]:
# ============================================================================
# EXPORT ALL DATASETS
# ============================================================================

timer_start('EXPORTING ALL DATASETS')

outputs_dir = ROOT / 'outputs'
files_created = []

# 1. Enriched official dataset (115 rows + news columns)
if len(df_enriched) > 0:
    enriched_path = outputs_dir / 'housing_projects_enriched.csv'
    df_enriched.to_csv(enriched_path, index=False)
    files_created.append(('housing_projects_enriched.csv', len(df_enriched), 'Official + news links'))
    print(f'Saved: {enriched_path}')

# 2. Comprehensive dataset (~203 rows)
if len(df_comprehensive) > 0:
    comprehensive_path = outputs_dir / 'housing_projects_comprehensive.csv'
    df_comprehensive.to_csv(comprehensive_path, index=False)
    files_created.append(('housing_projects_comprehensive.csv', len(df_comprehensive), 'All sources combined'))
    print(f'Saved: {comprehensive_path}')

# 3. Match results (for review)
if len(df_matches) > 0:
    matches_path = outputs_dir / 'match_results.csv'
    df_matches.to_csv(matches_path, index=False)
    files_created.append(('match_results.csv', len(df_matches), 'Fuzzy matching details'))
    print(f'Saved: {matches_path}')

# Summary table
print('\n' + '='*70)
print('FILES CREATED:')
print('-'*70)
for filename, rows, desc in files_created:
    print(f'  {filename:40} {rows:>6} rows  - {desc}')

files_created.append(('gellerman_integration_report.html', '-', 'Analysis report'))

timer_end()

EXPORTING ALL DATASETS
Started: 2026-02-26 19:57:43
Saved: /Users/johngage/berkeley-data/outputs/housing_projects_enriched.csv
Saved: /Users/johngage/berkeley-data/outputs/housing_projects_comprehensive.csv
Saved: /Users/johngage/berkeley-data/outputs/match_results.csv

FILES CREATED:
----------------------------------------------------------------------
  housing_projects_enriched.csv               115 rows  - Official + news links
  housing_projects_comprehensive.csv          156 rows  - All sources combined
  match_results.csv                           207 rows  - Fuzzy matching details
Completed: 2026-02-26 19:57:43
Duration: 0.01 seconds


## 8. Validation Checklist

In [16]:
# ============================================================================
# VALIDATION CHECKLIST
# ============================================================================

print('VALIDATION CHECKLIST')
print('='*70)

checks = []

# 1. Enriched dataset has 115 rows
enriched_ok = len(df_enriched) == 115 if len(df_enriched) > 0 else False
checks.append(('Enriched dataset has 115 rows', enriched_ok, len(df_enriched)))

# 2. Comprehensive dataset has more than official
comp_ok = len(df_comprehensive) > len(df_enriched) if len(df_comprehensive) > 0 else False
checks.append(('Comprehensive has more projects', comp_ok, len(df_comprehensive)))

# 3. No duplicate addresses in comprehensive
if len(df_comprehensive) > 0:
    dup_count = df_comprehensive['address_normalized'].dropna().duplicated().sum()
    no_dups = dup_count == 0
else:
    no_dups = False
    dup_count = 'N/A'
checks.append(('No duplicate addresses', no_dups, f'{dup_count} duplicates'))

# 4. All coordinates valid
if len(df_comprehensive) > 0:
    lat_valid = df_comprehensive['latitude'].between(37.845, 37.915) | df_comprehensive['latitude'].isna()
    lon_valid = df_comprehensive['longitude'].between(-122.325, -122.235) | df_comprehensive['longitude'].isna()
    invalid_coords = ((~lat_valid) | (~lon_valid)).sum()
    coords_ok = invalid_coords == 0
else:
    coords_ok = False
    invalid_coords = 'N/A'
checks.append(('All coordinates within Berkeley', coords_ok, f'{invalid_coords} invalid'))

# Print results
for check_name, passed, detail in checks:
    status = '[PASS]' if passed else '[FAIL]'
    print(f'{status} {check_name}: {detail}')

print('\n' + '='*70)
passed_count = sum(1 for _, p, _ in checks if p)
print(f'Passed: {passed_count}/{len(checks)} checks')

VALIDATION CHECKLIST
[PASS] Enriched dataset has 115 rows: 115
[PASS] Comprehensive has more projects: 156
[FAIL] No duplicate addresses: 3 duplicates
[PASS] All coordinates within Berkeley: 0 invalid

Passed: 3/4 checks


---

## Summary

This notebook:
- Loaded official (115) and Gellerman (~203) project data
- Performed fuzzy matching to link datasets
- Created enriched dataset with news coverage
- Created comprehensive dataset combining all sources
- Generated HTML integration report

**Outputs:**
- `outputs/housing_projects_enriched.csv` - 115 official + news
- `outputs/housing_projects_comprehensive.csv` - All ~203 projects
- `outputs/match_results.csv` - Matching details
- `outputs/gellerman_integration_report.html` - Analysis

**Next:** Run updated `D2_dashboard_data_export.ipynb` to deploy comprehensive data to Datasette.